# Name Entity Recognition using Deep Learning

Local version of the notebook for this repository.

Expected layout from the notebook working directory:

* `NERC-nn/`
* `lab_resources/DDI/data/{train,devel,test}`
* `lab_resources/DDI/util/evaluator.py`
* `requirements.txt`

Before running the notebook locally:

* create and activate a Python virtual environment
* install the dependencies with `%pip install -r requirements.txt`
* open the notebook from the repository root so the relative paths below resolve correctly


In [ ]:
# %pip install -r requirements.txt

In [1]:
from pathlib import Path

rootdir = Path.cwd()
rootdir

WindowsPath('d:/cosas_uni/master/Q2_25-26/MUD/task2')

Define the local paths to the code, data and evaluator:

In [34]:
utilsdir = str(rootdir / 'NERC-nn')
evaluatordir = str(rootdir / 'lab_resources' / 'DDI' / 'util')
traindir = str(rootdir / 'lab_resources' / 'DDI' / 'data' / 'train')
validationdir = str(rootdir / 'lab_resources' / 'DDI' / 'data' / 'devel')
testdir = str(rootdir / 'lab_resources' / 'DDI' / 'data' / 'test')
modelname = str(rootdir / 'model.keras')
outfile = str(rootdir / 'out.txt')

for path in [utilsdir, evaluatordir, traindir, validationdir, testdir]:
    print(path, '->', Path(path).exists())

d:\cosas_uni\master\Q2_25-26\MUD\task2\NERC-nn -> True
d:\cosas_uni\master\Q2_25-26\MUD\task2\lab_resources\DDI\util -> True
d:\cosas_uni\master\Q2_25-26\MUD\task2\lab_resources\DDI\data\train -> True
d:\cosas_uni\master\Q2_25-26\MUD\task2\lab_resources\DDI\data\devel -> True
d:\cosas_uni\master\Q2_25-26\MUD\task2\lab_resources\DDI\data\test -> True


In [35]:
import sys
sys.path.insert(1, str(utilsdir))
sys.path.insert(1, str(evaluatordir))


In [36]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

from contextlib import redirect_stdout

from tensorflow.keras import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, GRU, Embedding, Dense, TimeDistributed, Dropout, Bidirectional, concatenate, Softmax

import codemaps

import nltk
nltk.download('punkt')
nltk.download('punkt_tab', quiet=True)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pablo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [37]:
def build_network(codes) :
    # sizes
    n_words = codes.get_n_words()
    n_sufs = codes.get_n_sufs()
    n_labels = codes.get_n_labels()
    max_len = codes.maxlen


    ######################################
    inptW = Input(shape=(max_len,))
    inptS = Input(shape=(max_len,))
    model1 = Embedding(input_dim=n_words, output_dim=150,
                        input_length=max_len, mask_zero=False)(inptW)  # word embeddings
    model2 = Embedding(input_dim=n_sufs, output_dim=50,
                        input_length=max_len, mask_zero=False)(inptS)  # suf embeddings
    model1 = Dropout(0.1)(model1)
    model2 = Dropout(0.1)(model2)
    model = concatenate([model1,model2])
    y = Bidirectional(LSTM(units=200, return_sequences=True))(model)  #  biLSTM
    out = TimeDistributed(Dense(n_labels, activation=Softmax()))(y)

    return Model(
        inputs=[inptW,inptS], outputs=out
    )



def build_network2(codes):
    # sizes
    n_words = codes.get_n_words()
    n_lc_words = codes.get_n_lc_words()
    n_sufs = codes.get_n_sufs()
    n_shapes = codes.get_n_shapes()
    n_caps = codes.get_n_caps()
    n_nums = codes.get_n_nums()
    n_dashes = codes.get_n_dashes()
    n_gazetteer = codes.get_n_gazetteer()
    n_labels = codes.get_n_labels()
    max_len = codes.maxlen

    inptW = Input(shape=(max_len,)) # word input layer & embeddings
    embW = Embedding(input_dim=n_words, output_dim=100,
                    input_length=max_len, mask_zero=True)(inptW)

    inptLW = Input(shape=(max_len,)) # lowercased word input layer & embeddings
    embLW = Embedding(input_dim=n_lc_words, output_dim=100,
                    input_length=max_len, mask_zero=True)(inptLW)

    inptS = Input(shape=(max_len,))  # suffix input layer & embeddings
    embS = Embedding(input_dim=n_sufs, output_dim=50,
                    input_length=max_len, mask_zero=True)(inptS)

    inptC = Input(shape=(max_len,))  # capitalization feature
    embC = Embedding(input_dim=n_caps, output_dim=8,
                    input_length=max_len, mask_zero=True)(inptC)

    inptN = Input(shape=(max_len,))  # number feature
    embN = Embedding(input_dim=n_nums, output_dim=6,
                    input_length=max_len, mask_zero=True)(inptN)

    inptD = Input(shape=(max_len,))  # dash feature
    embD = Embedding(input_dim=n_dashes, output_dim=4,
                    input_length=max_len, mask_zero=True)(inptD)

    inptSh = Input(shape=(max_len,))  # token shape feature
    embSh = Embedding(input_dim=n_shapes, output_dim=20,
                    input_length=max_len, mask_zero=True)(inptSh)

    inptG = Input(shape=(max_len,))  # gazetteer feature
    embG = Embedding(input_dim=n_gazetteer, output_dim=8,
                    input_length=max_len, mask_zero=True)(inptG)

    dropW = Dropout(0.1)(embW)
    dropLW = Dropout(0.1)(embLW)
    dropS = Dropout(0.1)(embS)
    dropC = Dropout(0.05)(embC)
    dropN = Dropout(0.05)(embN)
    dropD = Dropout(0.05)(embD)
    dropSh = Dropout(0.1)(embSh)
    dropG = Dropout(0.05)(embG)
    drops = concatenate([dropW, dropLW, dropS, dropC, dropN, dropD, dropSh, dropG])

    # biLSTM
    bilstm = Bidirectional(LSTM(units=200, return_sequences=True,
                                dropout=0.1, recurrent_dropout=0.1))(drops)
    # output softmax layer
    out = TimeDistributed(Dense(n_labels, activation="softmax"))(bilstm)

    # build and compile model
    return Model([inptW, inptLW, inptS, inptC, inptN, inptD, inptSh, inptG], out)



In [38]:
# directory with files to process


# load train and validation data
traindata = codemaps.Dataset(traindir)

valdata = codemaps.Dataset(validationdir)

# create indexes from training data
max_len = 150
suf_len = 5
codes = codemaps.Codemaps(traindata, max_len, suf_len)

# encode datasets
# 1st ver -> [Xt,Xts] = codes.encode_words(traindata)
Xt = codes.encode_words(traindata)
Yt = codes.encode_labels(traindata)

# 1st ver -> [Xv,Xvs] = codes.encode_words(valdata)
Xv = codes.encode_words(valdata)
Yv = codes.encode_labels(valdata)

n_tags = codes.get_n_labels()
max_len = codes.maxlen

In [39]:
model = build_network2(codes)
model.compile(optimizer='adam' ,metrics=["accuracy"], loss="sparse_categorical_crossentropy")
model.build([(None, max_len)] * 8)

with redirect_stdout(sys.stderr) :
   model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_40      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_41      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_42      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_43      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_44      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_45      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_46      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_47      │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_40        │ (None, 150, 100)  │    967,600 │ input_layer_40[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_41        │ (None, 150, 100)  │    833,900 │ input_layer_41[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_42        │ (None, 150, 50)   │    247,850 │ input_layer_42[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_43        │ (None, 150, 8)    │         48 │ input_layer_43[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_44        │ (None, 150, 6)    │         24 │ input_layer_44[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_45        │ (None, 150, 4)    │         12 │ input_layer_45[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_46        │ (None, 150, 20)   │      6,000 │ input_layer_46[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_47        │ (None, 150, 8)    │         80 │ input_layer_47[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_40          │ (None, 150, 100)  │          0 │ embedding_40[0][

 Total params: 2,854,724 (10.89 MB)

 Trainable params: 2,854,724 (10.89 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
## --------- MAIN PROGRAM -----------
## --
## -- Usage:  train.py ../data/Train ../data/Devel  modelname
## --

# train model
with redirect_stdout(sys.stderr):
   model.fit(Xt, Yt, batch_size=32, epochs=10, validation_data=(Xv, Yv), verbose=1)

# save model and indexs
model.save(modelname)
codes.save(modelname)
#save_model_and_indexs(model, idx, modelname)

Epoch 1/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 41s 210ms/step - accuracy: 0.9182 - loss: 0.3592 - val_accuracy: 0.9703 - val_loss: 0.1202
Epoch 2/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 35s 208ms/step - accuracy: 0.9796 - loss: 0.0754 - val_accuracy: 0.9780 - val_loss: 0.0861
Epoch 3/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 34s 200ms/step - accuracy: 0.9889 - loss: 0.0398 - val_accuracy: 0.9811 - val_loss: 0.0751
Epoch 4/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 33s 194ms/step - accuracy: 0.9937 - loss: 0.0241 - val_accuracy: 0.9814 - val_loss: 0.0751
Epoch 5/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 33s 195ms/step - accuracy: 0.9950 - loss: 0.0177 - val_accuracy: 0.9802 - val_loss: 0.0795
Epoch 6/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 35s 203ms/step - accuracy: 0.9964 - loss: 0.0137 - val_accuracy: 0.9809 - val_loss: 0.0793
Epoch 7/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 34s 197ms/step - accuracy: 0.9967 - loss: 0.0106 - val_accuracy: 0.9807 - val_loss: 0.0805
Epoch 8/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 34s 199ms/step - accuracy: 0.9974 - loss: 0

# Predict

In [41]:
#import sys
import evaluator
import numpy as np

In [43]:
def output_entities(data, preds, outfile) :

   outf = open(outfile, 'w')
   for sid,tags in zip(data.sentence_ids(),preds) :
      inside = False
      for k in range(0,min(len(data.get_sentence(sid)),codes.maxlen)) :
         y = tags[k]
         token = data.get_sentence(sid)[k]

         if (y[0]=="B") :
             entity_form = token['form']
             entity_start = token['start']
             entity_end = token['end']
             entity_type = y[2:]
             inside = True
         elif (y[0]=="I" and inside) :
             entity_form += " "+token['form']
             entity_end = token['end']
         elif (y[0]=="O" and inside) :
             print(sid, str(entity_start)+"-"+str(entity_end), entity_form, entity_type, sep="|", file=outf)
             inside = False

      if inside : print(sid, str(entity_start)+"-"+str(entity_end), entity_form, entity_type, sep="|", file=outf)

   outf.close()

In [44]:
## --------- Evaluator -----------
def evaluation(datadir,outfile) :
   evaluator.evaluate("NER", datadir, outfile)


In [45]:
## --------- MAIN PROGRAM -----------
## --
## -- Usage:  baseline-NER.py target-dir
## --
## -- Extracts Drug NE from all XML files in target-dir
## --

datadir = validationdir

testdata = codemaps.Dataset(datadir)
# 1st ver -> [X,Xs] = codes.encode_words(testdata)
Xt = codes.encode_words(testdata)
Y = model.predict(Xt)
Y = [[codes.idx2label(np.argmax(w)) for w in s] for s in Y]

# extract entities
output_entities(testdata, Y, outfile)

# evaluate
evaluation(datadir,outfile)


45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step
                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
brand             288	  26	  86	 314	 374	91.7%	77.0%	83.7%
drug             1718	 134	 188	1852	1906	92.8%	90.1%	91.4%
drug_n              7	   9	  38	  16	  45	43.8%	15.6%	23.0%
group             577	 123	 110	 700	 687	82.4%	84.0%	83.2%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	77.7%	66.7%	70.3%
------------------------------------------------------------------------------
m.avg            2590	 292	 422	2882	3012	89.9%	86.0%	87.9%
m.avg(no class)  2678	 204	 334	2882	3012	92.9%	88.9%	90.9%
